# 0. 平坦地形时变电阻率模型

这一 notebook 只构建真值模型，不进行正演和反演。空间结构固定为 100 Ω·m 均匀背景中的矩形异常体；异常体平时为 50 Ω·m，位置不变，时间上先经历一次慢变化，恢复到 50 Ω·m 后再出现两个相邻 pulse，事件峰值达到 400 Ω·m。最终只生成一个连续时变模型。

## 1. 路径与基础参数

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

cwd = Path.cwd().resolve()
root = next(path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").exists())
output_dir = root / "data/test_model/0_flat_model"
output_dir.mkdir(parents=True, exist_ok=True)

background_rho = 100.0
baseline_anomaly_rho = 50.0
peak_anomaly_rho = 400.0
anomaly_bounds = {"x_min": 95.0, "x_max": 140.0, "depth_min": 12.0, "depth_max": 24.0}
segment_times = 25
n_times = 2 * segment_times - 1
model_name = "slow_then_double"

print("project root:", root)
print("output directory:", output_dir)

## 2. 构建平坦真值网格

水平方向采用 2.5 m 网格，范围为 0–235 m。垂向采用逐渐变粗的层厚，总深度 60 m。这里是真值/正演模型网格；后续反演会单独生成不一致的三角形参数网格。

In [ ]:
x_edges = np.arange(0.0, 235.0 + 2.5, 2.5, dtype=float)
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
layer_thickness = np.asarray([1.0] * 8 + [2.0] * 6 + [4.0] * 10, dtype=float)
z_edges = -np.concatenate(([0.0], np.cumsum(layer_thickness)))
z_centers = 0.5 * (z_edges[:-1] + z_edges[1:])
depth_centers = -z_centers

xx, depth = np.meshgrid(x_centers, depth_centers)
anomaly_mask = (
    (xx >= anomaly_bounds["x_min"])
    & (xx <= anomaly_bounds["x_max"])
    & (depth >= anomaly_bounds["depth_min"])
    & (depth <= anomaly_bounds["depth_max"])
)
electrode_x = np.arange(0.0, 235.0 + 5.0, 5.0, dtype=float)
electrode_z = np.zeros_like(electrode_x)

print("model cells:", anomaly_mask.size, "=", anomaly_mask.shape)
print("depth:", -z_edges[-1], "m")
print("electrodes:", electrode_x.size)
print("anomaly cells:", int(anomaly_mask.sum()))

## 3. 定义一条连续时间曲线

前25个时间点为 slow change；它恢复到异常体静态值 50 Ω·m 的时刻同时作为 double event 段的起点，因此总长度为49个时间点。曲线值从0到1，再通过对数插值映射为50–400 Ω·m；异常体外的背景始终为100 Ω·m。

In [ ]:
time_index = np.arange(n_times, dtype=np.int32)
time_normalized = np.linspace(0.0, 1.0, n_times, dtype=float)
segment_time = np.linspace(0.0, 1.0, segment_times, dtype=float)
segment_boundary_index = segment_times - 1

slow = np.sin(np.pi * segment_time) ** 2
double_event = np.exp(-0.5 * ((segment_time - 0.375) / 0.055) ** 2) + np.exp(
    -0.5 * ((segment_time - 0.625) / 0.055) ** 2
)

slow[[0, -1]] = 0.0
double_event[[0, -1]] = 0.0
slow /= slow.max()
double_event /= double_event.max()
curve_values = np.concatenate((slow, double_event[1:]))
anomaly_resistivity = baseline_anomaly_rho * np.exp(
    np.log(peak_anomaly_rho / baseline_anomaly_rho) * curve_values
)

print("model:", model_name)
print("time steps:", n_times, "| segment boundary:", segment_boundary_index)
print(
    f"anomaly range: {anomaly_resistivity.min():.1f}–{anomaly_resistivity.max():.1f} ohm-m"
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
ax.axvspan(
    0, segment_boundary_index, color="#277da1", alpha=0.08, label="Slow-change segment"
)
ax.axvspan(
    segment_boundary_index,
    n_times - 1,
    color="#7b2cbf",
    alpha=0.08,
    label="Double-event segment",
)
ax.plot(
    time_index,
    anomaly_resistivity,
    marker="o",
    markersize=3.5,
    linewidth=2.2,
    color="#343a40",
    label="Combined model",
)
ax.axvline(segment_boundary_index, color="0.45", linewidth=1.0, linestyle=":")
ax.axhline(
    baseline_anomaly_rho,
    color="#2a9d8f",
    linewidth=1.2,
    linestyle="--",
    label="Anomaly baseline (50)",
)
ax.axhline(
    background_rho,
    color="0.5",
    linewidth=1.0,
    linestyle=":",
    label="Host background (100)",
)
ax.set(xlabel="Time step", ylabel="Anomaly resistivity (ohm-m)", xlim=(0, n_times - 1))
ax.grid(alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.savefig(output_dir / "temporal_curves.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. 将时间曲线写入固定矩形异常体

In [ ]:
models = np.full(
    (n_times, z_centers.size, x_centers.size),
    background_rho,
    dtype=np.float64,
)
models[:, anomaly_mask] = anomaly_resistivity[:, None]

assert models.shape == (49, 24, 94)
assert np.all(np.isfinite(models)) and np.all(models > 0.0)
assert np.allclose(models[:, ~anomaly_mask], background_rho)
assert np.allclose(models[0, anomaly_mask], baseline_anomaly_rho)
assert np.allclose(models[segment_boundary_index, anomaly_mask], baseline_anomaly_rho)
assert np.allclose(models[-1, anomaly_mask], baseline_anomaly_rho)
assert np.isclose(anomaly_resistivity.max(), peak_anomaly_rho)
print("model array shape:", models.shape)
print("model range:", float(models.min()), float(models.max()), "ohm-m")

## 5. 检查空间模型快照

依次显示初始、慢变化峰值、段间恢复、双事件第一个峰、两峰低谷、第二个峰和最终恢复。纵轴显示完整的60 m模型深度。

In [ ]:
snapshot_ids = np.asarray([0, 12, 24, 33, 36, 39, 48], dtype=int)
fig, axes = plt.subplots(
    1,
    snapshot_ids.size,
    figsize=(18, 3.2),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
norm = LogNorm(vmin=30.0, vmax=600.0)
image = None
for col, step in enumerate(snapshot_ids):
    ax = axes[col]
    image = ax.pcolormesh(
        x_edges, z_edges, models[step], shading="flat", cmap="turbo", norm=norm
    )
    ax.scatter(electrode_x, electrode_z, s=7, c="black", zorder=3)
    ax.set_ylim(-60.0, 2.0)
    ax.set_title(f"t={step} | {anomaly_resistivity[step]:.0f} ohm-m")
    ax.set_xlabel("X (m)")
axes[0].set_ylabel("Elevation (m)")
fig.colorbar(image, ax=axes, label="Resistivity (ohm-m)", shrink=0.88)
fig.savefig(output_dir / "model_snapshots.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. 保存模型和元数据

主模型保存为 `resistivity.npy`，数组形状为 `(time, z, x)`，垂向使用从浅到深的顺序。

In [ ]:
for obsolete_name in (
    "slow_resistivity_top_to_bottom.npy",
    "fast_pulse_resistivity_top_to_bottom.npy",
    "double_event_resistivity_top_to_bottom.npy",
    "slow_then_double_resistivity_top_to_bottom.npy",
    "flat_model_bundle.npz",
):
    (output_dir / obsolete_name).unlink(missing_ok=True)
np.save(output_dir / "resistivity.npy", models)

metadata = {
    "schema_version": 5,
    "description": "60 m deep flat 100 ohm-m host with a 50 ohm-m rectangular anomaly at 12-24 m; slow change then a nearby double event reaches 400 ohm-m.",
    "resistivity_unit": "ohm-m",
    "coordinate_unit": "m",
    "array_vertical_order": "top_to_bottom",
    "model_shape": list(models.shape),
    "model_name": model_name,
    "time_steps": int(n_times),
    "time_segments": {
        "slow_change": [0, int(segment_boundary_index)],
        "double_event": [int(segment_boundary_index), int(n_times - 1)],
    },
    "background_resistivity": background_rho,
    "baseline_anomaly_resistivity": baseline_anomaly_rho,
    "peak_anomaly_resistivity": peak_anomaly_rho,
    "anomaly_bounds": anomaly_bounds,
    "grid": {
        "nx": int(x_centers.size),
        "nz": int(z_centers.size),
        "x_min": float(x_edges.min()),
        "x_max": float(x_edges.max()),
        "depth": float(-z_edges.min()),
    },
    "electrodes": {"count": int(electrode_x.size), "spacing": 5.0},
}
(output_dir / "metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print(json.dumps(metadata, indent=2))